In [1]:
import pandas as pd
import os

supplementary_dir = "results/Supplementary_Files"

os.makedirs(supplementary_dir, exist_ok=True)

In [2]:
def build_supplementary_table(
    mwu_path,
    delong_path,
    variant_column,
    top_column,
    second_column,
    variant_names,
    pair_column,
):
    metric_names = {
        "P_Value": "MWU-pvalue",
        "Adjusted_P_Value": "MWU-adjusted-pvalue",
        "ROC_AUC": "ROC-AUC",
        "PR_AUC": "PR-AUC",
    }

    mwu_df = pd.read_csv(mwu_path, sep="\t")
    delong_df = pd.read_csv(delong_path, sep="\t")

    mwu_metrics = mwu_df.reindex(
        columns=["Dataset", "TF", variant_column, *metric_names]
    )
    ordered_columns = [
        (metric, variant) for variant in variant_names for metric in metric_names
    ]

    variant_values = (
        mwu_metrics[mwu_metrics[variant_column].isin(variant_names)]
        .set_index(["Dataset", "TF", variant_column])[list(metric_names)]
        .unstack(variant_column)
        .reindex(columns=pd.MultiIndex.from_tuples(ordered_columns))
    )
    variant_values.columns = [
        f"{variant_names[variant]}_{metric_names[metric]}"
        for metric, variant in variant_values.columns
    ]

    summary = delong_df[
        [
            "Dataset",
            "TF",
            top_column,
            second_column,
            "DeLong_P_Value",
            "DeLong_P_Value_FDR_BH",
        ]
    ].copy()
    summary[pair_column] = (
        summary[top_column].map(variant_names)
        + ", "
        + summary[second_column].map(variant_names)
    )
    summary = summary.drop(columns=[top_column, second_column])

    result = (
        variant_values.reset_index()
        .merge(summary, on=["Dataset", "TF"], how="left")
        .rename(
            columns={
                "DeLong_P_Value": "DeLongs-pvalue",
                "DeLong_P_Value_FDR_BH": "DeLongs-adjusted-pvalue",
            }
        )
    )

    final_columns = (
        ["Dataset", "TF"]
        + list(variant_values.columns)
        + [
            pair_column,
            "DeLongs-pvalue",
            "DeLongs-adjusted-pvalue",
        ]
    )
    return result[final_columns]

## 1. Methods Supplementary File

In [3]:
method_names = {
    "z-aggregate": "z-aggregate",
    "viper": "viper",
    "ulm": "ulm",
    "zscore": "zscore",
}

methods_delongs_df = build_supplementary_table(
    mwu_path="results/Methods_MWU-Delongs/MWU_merged.tsv",
    delong_path="results/Methods_MWU-Delongs/DeLong_top2_merged.tsv",
    variant_column="Method",
    top_column="Top_Method",
    second_column="Second_Method",
    variant_names=method_names,
    pair_column="Top, Second",
)

methods_delongs_df.to_csv(
    f"{supplementary_dir}/Methods_Delongs_Comparison_Table.tsv", sep="\t", index=False
)
methods_delongs_df

,Dataset,TF,z-aggregate_MWU-pvalue,z-aggregate_MWU-adjusted-pvalue,z-aggregate_ROC-AUC,z-aggregate_PR-AUC,viper_MWU-pvalue,viper_MWU-adjusted-pvalue,viper_ROC-AUC,viper_PR-AUC,...,ulm_MWU-adjusted-pvalue,ulm_ROC-AUC,ulm_PR-AUC,zscore_MWU-pvalue,zscore_MWU-adjusted-pvalue,zscore_ROC-AUC,zscore_PR-AUC,"Top, Second",DeLongs-pvalue,DeLongs-adjusted-pvalue
0,FrangiehIzar2021_RNA,CCND1,0.483509,0.863095,0.500478,0.010632,0.808274,0.889101,0.489928,0.010695,...,0.270035,0.512746,0.011603,0.066495,0.132991,0.517362,0.011808,NaN,NaN,NaN
1,FrangiehIzar2021_RNA,CDKN1A,0.395770,0.863095,0.502392,0.018090,0.213266,0.426533,0.507197,0.018847,...,0.265413,0.510606,0.018909,0.044510,0.097922,0.515392,0.019170,"zscore, ulm",0.000002,0.000037
2,FrangiehIzar2021_RNA,CDKN2A,0.632405,0.993780,0.496655,0.014714,0.000730,0.002294,0.531478,0.016856,...,0.001151,0.533401,0.016857,0.000043,0.000158,0.538837,0.017075,"zscore, ulm",0.003797,0.021835
3,FrangiehIzar2021_RNA,DNMT1,0.890240,0.999738,0.480461,0.005260,0.448481,0.657771,0.502061,0.006001,...,0.577921,0.502395,0.005942,0.235551,0.345475,0.511469,0.006025,NaN,NaN,NaN
4,FrangiehIzar2021_RNA,E2F1,0.999738,0.999738,0.458327,0.008719,0.890574,0.932983,0.485223,0.010111,...,0.969018,0.480277,0.009854,0.872095,0.913624,0.486344,0.009990,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
664,WesselsSatija2023,HDAC3,0.993613,0.993613,0.417969,0.148876,0.002125,0.005667,0.594232,0.287770,...,0.003241,0.610391,0.258064,0.004666,0.012443,0.585686,0.235450,"ulm, viper",0.311268,0.577352
665,WesselsSatija2023,MYB,0.005282,0.010564,0.708166,0.141106,0.144952,0.193270,0.586219,0.043269,...,0.149962,0.607364,0.042851,0.131611,0.200419,0.591141,0.041517,"z-aggregate, ulm",0.135269,0.319893
666,WesselsSatija2023,SSRP1,0.200903,0.292318,0.545100,0.082024,0.200237,0.228842,0.545253,0.079308,...,0.200419,0.550222,0.081574,0.175366,0.200419,0.550222,0.081574,NaN,NaN,NaN
667,WesselsSatija2023,SUPT16H,0.000312,0.002463,0.598658,0.294250,0.001132,0.004530,0.588238,0.280059,...,0.003631,0.590138,0.287285,0.000908,0.007263,0.590138,0.287285,"z-aggregate, ulm",0.128115,0.310174


## 2. Weights Supplementary File

In [4]:
weight_names = {
    "CORRELATION": "Correlation",
    "NONZERORATE": "Nonzero-Rate",
    "SPECIFICITY": "Specificity",
    "UNIFORM": "Uniform",
}

weights_delongs_df = build_supplementary_table(
    mwu_path="results/Weights_MWU-Delongs/MWU_merged.tsv",
    delong_path="results/Weights_MWU-Delongs/DeLong_top2_merged.tsv",
    variant_column="Weight",
    top_column="Top_Weight",
    second_column="Second_Weight",
    variant_names=weight_names,
    pair_column="Top, Second",
)

weights_delongs_df.to_csv(
    f"{supplementary_dir}/Weights_Delongs_Comparison_Table.tsv", sep="\t", index=False
)
weights_delongs_df

,Dataset,TF,Correlation_MWU-pvalue,Correlation_MWU-adjusted-pvalue,Correlation_ROC-AUC,Correlation_PR-AUC,Nonzero-Rate_MWU-pvalue,Nonzero-Rate_MWU-adjusted-pvalue,Nonzero-Rate_ROC-AUC,Nonzero-Rate_PR-AUC,...,Specificity_MWU-adjusted-pvalue,Specificity_ROC-AUC,Specificity_PR-AUC,Uniform_MWU-pvalue,Uniform_MWU-adjusted-pvalue,Uniform_ROC-AUC,Uniform_PR-AUC,"Top, Second",DeLongs-pvalue,DeLongs-adjusted-pvalue
0,FrangiehIzar2021_RNA,CCND1,0.868610,0.909973,0.487059,0.010250,0.397406,0.794812,0.503005,0.010603,...,0.986392,0.491187,0.010408,0.483509,0.863095,0.500478,0.010632,NaN,NaN,NaN
1,FrangiehIzar2021_RNA,CDKN1A,0.295440,0.722187,0.504865,0.017839,0.451176,0.827156,0.501110,0.017962,...,0.986392,0.497833,0.017796,0.395770,0.863095,0.502392,0.018090,NaN,NaN,NaN
2,FrangiehIzar2021_RNA,CDKN2A,0.825418,0.909973,0.490740,0.014514,0.725301,0.999978,0.494079,0.014723,...,0.986392,0.484075,0.014338,0.632405,0.993780,0.496655,0.014714,NaN,NaN,NaN
3,FrangiehIzar2021_RNA,DNMT1,0.348779,0.767313,0.506184,0.006056,0.725656,0.999978,0.490456,0.005402,...,0.986392,0.474278,0.005256,0.890240,0.999738,0.480461,0.005260,NaN,NaN,NaN
4,FrangiehIzar2021_RNA,E2F1,0.998515,0.998515,0.464298,0.008930,0.999978,0.999978,0.450931,0.008564,...,0.986392,0.476155,0.009144,0.999738,0.999738,0.458327,0.008719,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
664,WesselsSatija2023,HDAC3,0.003269,0.008716,0.589619,0.236734,0.976045,0.976045,0.434834,0.155383,...,0.970633,0.437733,0.154477,0.993613,0.993613,0.417969,0.148876,"Correlation, Specificity",0.012308,0.049606
665,WesselsSatija2023,MYB,0.009888,0.019776,0.689756,0.056897,0.003681,0.007363,0.718192,0.118490,...,0.011896,0.712906,0.088185,0.005282,0.010564,0.708166,0.141106,"Nonzero-Rate, Specificity",0.933484,0.966174
666,WesselsSatija2023,SSRP1,0.098409,0.131212,0.569408,0.088952,0.343924,0.458565,0.521633,0.073554,...,0.428262,0.524996,0.079636,0.200903,0.292318,0.545100,0.082024,NaN,NaN,NaN
667,WesselsSatija2023,SUPT16H,0.001321,0.008716,0.586719,0.288139,0.000327,0.001309,0.598296,0.288993,...,0.003730,0.595474,0.292836,0.000312,0.002463,0.598658,0.294250,"Uniform, Nonzero-Rate",0.971238,0.988638


## 3. Priors Supplementary Files

Separate tables are generated for the AllAvailableTFs and CommonTFsOnly analyses. Both use the corresponding With-Ensemble DeLong top-two comparison.

In [5]:
prior_names = {
    "causalpath": "CausalPath",
    "collectri": "CollecTRI",
    "dorothea": "DoRothEA",
    "ensemble": "Ensemble",
}

prior_supplementary_configs = {
    "AllAvailableTFs": {
        "mwu_filename": "MWU_merged_AllAvailableTFs.tsv",
        "delong_filename": "DeLong_top2_With-Ensemble_AllAvailableTFs.tsv",
    },
    "CommonTFsOnly": {
        "mwu_filename": "MWU_merged_CommonTFsOnly.tsv",
        "delong_filename": "DeLong_top2_With-Ensemble_CommonTFsOnly.tsv",
    },
}

prior_table_summary = []
for tf_set, filenames in prior_supplementary_configs.items():
    priors_delongs_df = build_supplementary_table(
        mwu_path=f"results/Priors_MWU-Delongs/{filenames['mwu_filename']}",
        delong_path=f"results/Priors_MWU-Delongs/{filenames['delong_filename']}",
        variant_column="Prior",
        top_column="Top_Prior",
        second_column="Second_Prior",
        variant_names=prior_names,
        pair_column="Top, Second",
    )
    output_path = f"{supplementary_dir}/Priors_Delongs_Comparison_Table_{tf_set}.tsv"
    priors_delongs_df.to_csv(output_path, sep="\t", index=False)
    prior_table_summary.append({"TF set": tf_set, "Rows": len(priors_delongs_df), "File": output_path})

pd.DataFrame(prior_table_summary)

,Dataset,TF,CausalPath-Priors_MWU-pvalue,CausalPath-Priors_MWU-adjusted-pvalue,CausalPath-Priors_ROC-AUC,CausalPath-Priors_PR-AUC,Collectri_MWU-pvalue,Collectri_MWU-adjusted-pvalue,Collectri_ROC-AUC,Collectri_PR-AUC,...,Dorothea_MWU-adjusted-pvalue,Dorothea_ROC-AUC,Dorothea_PR-AUC,Ensemble_MWU-pvalue,Ensemble_MWU-adjusted-pvalue,Ensemble_ROC-AUC,Ensemble_PR-AUC,"Top, Second",DeLongs-pvalue,DeLongs-adjusted-pvalue
0,FrangiehIzar2021_RNA,CCND1,0.483509,0.863095,0.500478,0.010632,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.463669,0.999997,0.501054,0.010644,NaN,NaN,NaN
1,FrangiehIzar2021_RNA,CDKN1A,0.395770,0.863095,0.502392,0.018090,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.395770,0.999997,0.502392,0.018090,NaN,NaN,NaN
2,FrangiehIzar2021_RNA,CDKN2A,0.632405,0.993780,0.496655,0.014714,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.632405,0.999997,0.496655,0.014714,NaN,NaN,NaN
3,FrangiehIzar2021_RNA,DNMT1,0.890240,0.999738,0.480461,0.005260,0.719821,0.999850,0.490733,0.005516,...,NaN,NaN,NaN,0.780128,0.999997,0.487705,0.005447,NaN,NaN,NaN
4,FrangiehIzar2021_RNA,E2F1,0.999738,0.999738,0.458327,0.008719,0.999850,0.999850,0.456550,0.008667,...,0.999993,0.447817,0.008514,0.999997,0.999997,0.445242,0.008471,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1295,WesselsSatija2023,MYB,0.005282,0.010564,0.708166,0.141106,0.011399,0.028497,0.685381,0.064225,...,0.001480,0.774699,0.107400,0.002124,0.006371,0.732774,0.083839,"Dorothea, Ensemble",0.376726,0.376726
1296,WesselsSatija2023,SMARCA5,NaN,NaN,NaN,NaN,0.009540,0.028497,0.583364,0.208826,...,NaN,NaN,NaN,0.009540,0.022896,0.583364,0.208826,NaN,NaN,NaN
1297,WesselsSatija2023,SSRP1,0.200903,0.292318,0.545100,0.082024,0.981249,0.981249,0.388167,0.064000,...,NaN,NaN,NaN,0.896950,0.978491,0.432044,0.066467,NaN,NaN,NaN
1298,WesselsSatija2023,SUPT16H,0.000312,0.002463,0.598658,0.294250,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.000312,0.001875,0.598658,0.294250,NaN,NaN,NaN
